In [81]:
import pandas as pd
from operator import itemgetter
import networkx as nx

In [82]:
data = pd.read_csv('pre_survey.csv')

### Nodes

In [83]:
nodes_df = data[['ID','Name']] #only need these columns, we fit everything to name for ease (only 40 people)
nodes_df = nodes_df.rename(columns={'Name':'Label'})
print("There are", len(nodes_df), "nodes.")

There are 40 nodes.


In [84]:
nodes_df.to_csv('nodes.csv', index=False)

### Edges

In [85]:
working = data.iloc[:,4:] #removes this column
working = working.set_index('Name').drop(columns=['NetID','Last modified time']) #set index for stacking, remove other cols
working = working.stack().rename_axis(['Source','Target']).reset_index() #makes into a stack from matrix
edges = working.rename(columns={0:'Weight'}) #rename columns
edges = edges[edges['Target'].isin(nodes_df['Label'])] #only include targest in the club
edges.head()

,Source,Target,Weight
0,Nikhil Chinchalkar,Nikhil Chinchalkar,I am this person
1,Nikhil Chinchalkar,Jason Wang,I speak with them at least once a week
2,Nikhil Chinchalkar,Rithya Sriram,I speak with them at least once a week
3,Nikhil Chinchalkar,Carina Lau,I speak with them at least once a week
4,Nikhil Chinchalkar,Jenny Williams,I speak with them at least once a week


In [86]:
#TODO: @Emi figure out what to do with Isabella Guan. She was not in a column but responded to the survey. Can avg/throw out/dup her responses.
len(edges['Source'].unique()), len(edges['Target'].unique()) #sanity check, should be 40 people still

(40, 39)

In [87]:
edges['Weight'].unique() #used for mapping weights below

array(['I am this person', 'I speak with them at least once a week',
       "I've spoken to them more than once before",
       'I recognize their face/name', "I've spoken to them once before",
       "I've never seen/heard of this person before",
       'I speak with them everyday'], dtype=object)

In [88]:
weights_map = {'I am this person':0,
               'I speak with them everyday':6,
               'I speak with them at least once a week':5,
               "I've spoken to them more than once before":4,
               "I've spoken to them once before":3,
               'I recognize their face/name':2,
               "I've never seen/heard of this person before":0} #subject to change

In [89]:
edges['Weight'] = edges['Weight'].map(lambda x: weights_map[x])

In [90]:
edges.to_csv('edges.csv', index=False)

### Demographics

In [94]:
demographics = pd.read_csv('demographics.csv')
demographics = pd.merge(demographics, nodes_df, left_on='Full Name', right_on='Label', how='right')
demographics.to_csv('node_demographics.csv')